In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_curve,
    auc,
    confusion_matrix,
)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from matplotlib import rcParams
from tqdm import tqdm
import shap
import time

In [ ]:
# ===================== DEVICE SETUP ===================== #
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")

In [ ]:
# ===================== PATHS AND CONFIGURATION ===================== #
model_name = "convnext"
OUTPUT_PATH = f"1_Feature_Extraction/{model_name}"
RESULTS_PATH = os.path.join("9_Ablation_Study")
os.makedirs(RESULTS_PATH, exist_ok=True)


# Load feature data
try:
    train_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"train_features_{model_name}.csv"))
    test_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"test_features_{model_name}.csv"))
    print(f"Loaded {len(train_df)} training and {len(test_df)} testing samples")
except FileNotFoundError:
    print(f"{model_name} features not found, attempting to use available features")
    train_df = pd.read_csv(os.path.join(f"1_Feature_Extraction/{model_name}", f"train_features_{model_name}.csv"))
    test_df = pd.read_csv(os.path.join(f"1_Feature_Extraction/{model_name}", f"test_features_{model_name}.csv"))
    print(f"Using {model_name} features with {len(train_df)} training and {len(test_df)} testing samples")

In [ ]:
# Extract features and labels
feature_columns = [col for col in train_df.columns if col.startswith("feat_")]
X_train = train_df[feature_columns].values
y_train = train_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values
X_test = test_df[feature_columns].values
y_test = test_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
N_CLASSES = len(CLASSES)  # Should be 4
print(f"Number of classes: {N_CLASSES}")

In [ ]:
# Plot settings
rcParams["font.family"] = "Times New Roman"
rcParams["axes.titlesize"] = 28
rcParams["axes.titlepad"] = 20
rcParams["axes.labelsize"] = 23
rcParams["xtick.labelsize"] = 18
rcParams["ytick.labelsize"] = 18
rcParams["legend.fontsize"] = 16
rcParams["lines.linewidth"] = 3
rcParams["axes.linewidth"] = 2

In [ ]:
# ===================== FEATURE SELECTION USING SHAP ===================== #
def apply_shap(X_train, X_test, n_features):
    print(f"Applying SHAP to select {n_features} features...")
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
    rf_model.fit(X_train, y_train)
    feature_importance = rf_model.feature_importances_
    selected_indices = np.argsort(feature_importance)[::-1][:n_features]
    selected_indices = np.array(selected_indices, dtype=int)
    X_train_selected = X_train[:, selected_indices]
    X_test_selected = X_test[:, selected_indices]
    return X_train_selected, X_test_selected, selected_indices

In [ ]:
# ===================== DATASET DEFINITION ===================== #
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
# ===================== MODEL VARIATIONS FOR ABLATION STUDY ===================== #
# Original Attention Module
class Attention(nn.Module):
    def __init__(self, feature_dim):
        super(Attention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
            nn.Softmax(dim=1),
        )

    def forward(self, x):
        weights = self.attention(x)
        return (x * weights).sum(dim=1)

In [ ]:
# Original AttGRU Model
class AttGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(AttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [ ]:
# Variation 1: Without Attention
class GRUOnly(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(GRUOnly, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        _, h = self.gru(x)
        x = h.squeeze(0)  # Use the final hidden state
        x = self.fc(x)
        return x

In [ ]:
# Variation 2: LSTM instead of GRU
class AttLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(AttLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.lstm(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [ ]:
# Variation 3: Bidirectional GRU
class BidirectionalAttGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(BidirectionalAttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = Attention(hidden_dim * 2)  # Double size for bidirectional
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [ ]:
# Variation 4: Simple MLP (No GRU, No Attention)
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(SimpleMLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.mlp(x)

In [ ]:
# Variation 5: Deeper GRU with Attention
class DeepAttGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(DeepAttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.3)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [ ]:
# ===================== TRAINING AND EVALUATION ===================== #
def train_model(model, train_loader, test_loader, model_name,
               learning_rate=0.001, num_epochs=100, weight_decay=1e-5):
    criterion = nn.CrossEntropyLoss()
    
    # Initialize AdamW optimizer with fixed learning rate
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    train_losses, test_losses, train_accs, test_accs = [], [], [], []
    
    # For measuring training time
    start_time = time.time()
    
    for epoch in tqdm(range(num_epochs), desc=f"Training {model_name} with AdamW LR={learning_rate}"):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_loss = test_loss / len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)

    training_time = time.time() - start_time
    
    print(f"Training completed in {training_time:.2f} seconds")
    
    return train_losses, test_losses, train_accs, test_accs, training_time

In [ ]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        outputs = model(X_test_tensor)
        _, y_pred = outputs.max(1)
        y_pred = y_pred.cpu().numpy()
        y_prob = torch.softmax(outputs, dim=1).cpu().numpy()

    metrics = {}
    metrics["ACC"] = accuracy_score(y_test, y_pred)
    metrics["AUC"] = roc_auc_score(y_test, y_prob, multi_class="ovr")
    metrics["PRE"] = precision_score(y_test, y_pred, average="macro")
    metrics["SN"] = recall_score(y_test, y_pred, average="macro")
    
    # Calculate specificity for multiclass
    cm = confusion_matrix(y_test, y_pred)
    specificity_scores = []
    for i in range(N_CLASSES):
        # True negatives are all elements of the confusion matrix except for the current class
        tn = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + cm[i, i]
        fp = np.sum(cm[:, i]) - cm[i, i]
        # Avoid division by zero
        if (tn + fp) == 0:
            specificity_scores.append(0)
        else:
            specificity_scores.append(tn / (tn + fp))
    metrics["SP"] = np.mean(specificity_scores)
    
    metrics["F1"] = f1_score(y_test, y_pred, average="macro")
    metrics["MCC"] = matthews_corrcoef(y_test, y_pred)
    return metrics, y_prob, y_pred

In [ ]:
def plot_roc(y_test, y_prob, model_name):
    y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    plt.figure(figsize=(8, 8))

    for i in range(N_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f"{CLASSES[i]} (AUC = {roc_auc[i]:.2f})")

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"{model_name} - ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    
    # Save as PNG
    plt.savefig(os.path.join(RESULTS_PATH, f"{model_name}_roc_curve.png"), dpi=1000, bbox_inches="tight")
    # Save as PDF
    plt.savefig(os.path.join(RESULTS_PATH, f"{model_name}_roc_curve.pdf"), format='pdf', bbox_inches="tight")
    plt.close()

In [ ]:
def plot_learning_curves(train_losses, test_losses, train_accs, test_accs, model_name):
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title(f'{model_name} - Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(test_accs, label='Test Accuracy')
    plt.title(f'{model_name} - Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    # Save as PNG
    plt.savefig(os.path.join(RESULTS_PATH, f"{model_name}_learning_curves.png"), dpi=1000, bbox_inches="tight")
    # Save as PDF
    plt.savefig(os.path.join(RESULTS_PATH, f"{model_name}_learning_curves.pdf"), format='pdf', bbox_inches="tight")
    plt.close()

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
print("Starting ablation study for AttGRU model variations...")

# Define a fixed number of features to use
n_features = 800
print(f"\n===== Selecting {n_features} features using SHAP =====")
X_train_shap, X_test_shap, shap_indices = apply_shap(X_train, X_test, n_features=n_features)

In [ ]:
# Define model variations for ablation study
model_variations = {
    "AttGRU (Original)": AttGRU,
    "GRU without Attention": GRUOnly,
    "LSTM with Attention": AttLSTM,
    "Bidirectional AttGRU": BidirectionalAttGRU,
    "Simple MLP": SimpleMLP,
    "Deep AttGRU (2 layers)": DeepAttGRU
}

In [ ]:
# Fixed parameters
learning_rate = 0.001
num_epochs = 100
batch_size = 64

# Prepare datasets and dataloaders
train_dataset = FeatureDataset(X_train_shap, y_train)
test_dataset = FeatureDataset(X_test_shap, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Track results for all model variations
ablation_results = []

# For plotting comparative learning curves
all_train_losses = {}
all_test_losses = {}
all_train_accs = {}
all_test_accs = {}

In [ ]:
for model_name, model_class in model_variations.items():
    print(f"\n===== Evaluating {model_name} =====")
    
    # Initialize the current model variation
    model = model_class(input_dim=X_train_shap.shape[1]).to(device)
    
    # Count parameters
    parameter_count = count_parameters(model)
    print(f"Total trainable parameters: {parameter_count:,}")
    
    # Train the model
    train_losses, test_losses, train_accs, test_accs, training_time = train_model(
        model, 
        train_loader, 
        test_loader,
        model_name=model_name,
        learning_rate=learning_rate,
        num_epochs=num_epochs
    )
    
    # Store for comparative plots
    all_train_losses[model_name] = train_losses
    all_test_losses[model_name] = test_losses
    all_train_accs[model_name] = train_accs
    all_test_accs[model_name] = test_accs

    # Plot individual learning curves
    plot_learning_curves(train_losses, test_losses, train_accs, test_accs, model_name)

    # Evaluate the model
    metrics, y_prob, y_pred = evaluate_model(model, X_test_shap, y_test)
    
    # Plot ROC curve
    plot_roc(y_test, y_prob, model_name)
    
    # Save model
    torch.save(model.state_dict(), os.path.join(RESULTS_PATH, f"{model_name.replace(' ', '_')}_model.pth"))

    # Save results
    ablation_results.append(
        {
            "Model": model_name,
            "Parameters": parameter_count,
            "Training_Time": training_time,
            "ACC": metrics["ACC"],
            "AUC": metrics["AUC"],
            "PRE": metrics["PRE"],
            "SN": metrics["SN"],
            "SP": metrics["SP"],
            "F1": metrics["F1"],
            "MCC": metrics["MCC"],
        }
    )
    print(f"{model_name} - Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print(f"Training Time: {training_time:.2f} seconds")

In [ ]:
# Save comparison results to CSV
ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(os.path.join(RESULTS_PATH, "AttGRU_ablation_study_results.csv"), index=False)
print(f"Saved ablation study results to {os.path.join(RESULTS_PATH, 'AttGRU_ablation_study_results.csv')}")

# Plot comparison of metrics across models
plt.figure(figsize=(14, 8))
metrics_to_plot = ["ACC", "AUC", "SP", "SN", "F1", "MCC"]
model_names = [result["Model"] for result in ablation_results]
x = np.arange(len(model_names))
width = 0.15

In [ ]:
for i, metric in enumerate(metrics_to_plot):
    values = [result[metric] for result in ablation_results]
    plt.bar(x + i * width, values, width, label=metric)

In [ ]:
plt.xlabel("Model Variation")
plt.ylabel("Score")
plt.title("Performance Comparison of Model Variations")
plt.xticks(x + width * (len(metrics_to_plot) - 1) / 2, model_names, rotation=15, ha='right')
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_ablation_study_metrics.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_ablation_study_metrics.pdf"), format='pdf', bbox_inches="tight")
plt.close()

In [ ]:
# Plot comparative learning curves
plt.figure(figsize=(16, 12))

# Loss curves
plt.subplot(2, 2, 1)
for model_name, losses in all_train_losses.items():
    plt.plot(losses, label=model_name)
plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 2)
for model_name, losses in all_test_losses.items():
    plt.plot(losses, label=model_name)
plt.title('Test Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy curves
plt.subplot(2, 2, 3)
for model_name, accs in all_train_accs.items():
    plt.plot(accs, label=model_name)
plt.title('Training Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 4)
for model_name, accs in all_test_accs.items():
    plt.plot(accs, label=model_name)
plt.title('Test Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_ablation_study_learning_curves.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_ablation_study_learning_curves.pdf"), format='pdf', bbox_inches="tight")
plt.close()

In [ ]:
# Plot parameters vs performance
plt.figure(figsize=(14, 8))
parameters = [result["Parameters"] for result in ablation_results]
parameters_normalized = [p / max(parameters) for p in parameters]  # Normalize for bubble size

for i, metric in enumerate(["ACC", "AUC", "F1"]):
    plt.subplot(1, 3, i+1)
    metric_values = [result[metric] for result in ablation_results]
    training_times = [result["Training_Time"] for result in ablation_results]
    
    # Create bubble plot: x=Parameters, y=Metric, size=Training Time
    bubble_sizes = [100 + 1000 * (t / max(training_times)) for t in training_times]
    
    scatter = plt.scatter(parameters, metric_values, s=bubble_sizes, alpha=0.6)
    
    # Add model names as labels
    for j, model_name in enumerate(model_names):
        plt.annotate(model_name, 
                     (parameters[j], metric_values[j]),
                     textcoords="offset points",
                     xytext=(0, 10),
                     ha='center',
                     fontsize=8)
    
    plt.grid(True)
    plt.xlabel("Number of Parameters")
    plt.ylabel(metric)
    plt.title(f"{metric} vs Model Complexity")

plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_ablation_study_complexity.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_ablation_study_complexity.pdf"), format='pdf', bbox_inches="tight")
plt.close()